# Extra Lab — Floating-Point Error (Computational Error Demonstrations)

**Goal:** Build intuition for floating-point round-off and how it shows up in physics computations.

Key idea:
> Floating-point numbers are approximations (binary scientific notation), not real numbers.

You will see:
- Why `0.1 + 0.2 != 0.3` (sometimes)
- Why summation order matters
- Catastrophic cancellation
- Why naive numerical differentiation can fail
- Practical fixes: tolerances, stable formulas, Kahan summation


In [2]:
import math
import sys
from decimal import Decimal, getcontext


## 1) What is a float?

Python `float` is typically IEEE-754 double precision:
- ~16 decimal digits of precision
- finite binary representation

Let's check machine epsilon (spacing near 1.0):
\[
\epsilon \approx 2^{-52} \approx 2.22\times 10^{-16}
\]


In [3]:
sys.float_info.epsilon


2.220446049250313e-16

## 2) The classic: `0.1 + 0.2`

Because 0.1 and 0.2 are not exactly representable in binary,
their stored values are slightly off.

The result is usually:
- `0.1 + 0.2 = 0.30000000000000004`


In [4]:
x = 0.1 + 0.2
y = 0.3
x, y, (x == y), (x - y)


(0.30000000000000004, 0.3, False, 5.551115123125783e-17)

### Correct way: compare within tolerance

Physics mindset:
\[
|x-y| < \varepsilon
\]

Or use `math.isclose` (recommended).


In [5]:
abs(x - y) < 1e-12, math.isclose(x, y), math.isclose(x, y, rel_tol=1e-12, abs_tol=0.0)


(True, True, True)

## 3) Representation peek: `Decimal` vs float

`Decimal` stores base-10 decimals more "as written" (at a cost).
It’s useful for demos, not for fast scientific computing.

We'll compare float vs Decimal arithmetic.


In [ ]:
getcontext().prec = 50

xf = 0.1 + 0.2
xd = Decimal("0.1") + Decimal("0.2")

xf, xd


## 4) Summation order matters

If you add a **tiny** number to a **huge** number, the tiny one can vanish:
> the float has limited precision relative to the magnitude.

This is common in physics when accumulating:
- energy budgets
- integrals
- large datasets

We'll show two sums:
1) add small terms first (good)
2) add small terms after a huge term (bad)


In [ ]:
huge = 1e16
small = 1.0

(huge + small) - huge, (huge + 10*small) - huge


In [ ]:
# Build a list: one huge number + many small numbers
data = [1e16] + [1.0] * 1_000_000

sum_forward = sum(data)                 # huge first
sum_reverse = sum(reversed(data))       # small first

sum_forward, sum_reverse, (sum_reverse - sum_forward)


### Kahan Summation (compensated summation)

This algorithm reduces floating-point error in long sums.
It's useful when:
- you sum many numbers with varying magnitudes
- you want more stable accumulation

We'll implement a simple Kahan sum.


In [ ]:
def kahan_sum(values):
    total = 0.0
    c = 0.0  # compensation for lost low-order bits
    for v in values:
        y = v - c
        t = total + y
        c = (t - total) - y
        total = t
    return total

sum_kahan = kahan_sum(data)
sum_forward, sum_reverse, sum_kahan


## 5) Catastrophic cancellation

When subtracting two nearly equal numbers, most significant digits cancel,
and the relative error can blow up.

Classic example:
\[
\sqrt{1+\epsilon} - 1 \quad \text{for small }\epsilon
\]

Naive evaluation loses precision.
A stable alternative uses algebra:

\[
\sqrt{1+\epsilon} - 1 = \frac{\epsilon}{\sqrt{1+\epsilon}+1}
\]


In [ ]:
def naive(eps):
    return math.sqrt(1.0 + eps) - 1.0

def stable(eps):
    return eps / (math.sqrt(1.0 + eps) + 1.0)

for eps in [1e-2, 1e-6, 1e-10, 1e-14, 1e-16]:
    n = naive(eps)
    s = stable(eps)
    print(f"eps={eps: .0e}  naive={n: .20e}  stable={s: .20e}  diff={abs(n-s):.2e}")


### Physics connection

In orbital mechanics, thermodynamics, spectroscopy, etc., you often compute:
- differences between large energies
- potential minus kinetic
- nearby eigenvalues

Catastrophic cancellation is why **algebraic reformulation** matters.


## 6) Numerical differentiation and round-off

Finite differences:
\[
f'(x) \approx \frac{f(x+h)-f(x)}{h}
\]

If \(h\) is too large → truncation error.
If \(h\) is too small → subtraction + floating-point round-off dominates.

We'll demonstrate with:
\[
f(x) = \sin(x), \quad f'(x)=\cos(x)
\]


In [ ]:
def fd_derivative(f, x, h):
    return (f(x + h) - f(x)) / h

x = 1.234
true = math.cos(x)

hs = [10**(-k) for k in range(1, 18)]
errors = []
for h in hs:
    approx = fd_derivative(math.sin, x, h)
    errors.append(abs(approx - true))

list(zip(hs[:5], errors[:5]))[:5], (hs[-1], errors[-1])


In [ ]:
best = min(zip(hs, errors), key=lambda t: t[1])
best


### Why this matters

When you:
- estimate gradients (optimization, fitting)
- compute derivatives (fluid dynamics, radiative transfer)
- do numerical methods

You must choose step sizes and methods carefully.

This is also why libraries (NumPy/SciPy) implement stable algorithms.


## 7) Practical rules of thumb

1) **Never** compare floats with `==`  
   Use `math.isclose` or `abs(a-b) < tol`.

2) When summing many values with different magnitudes:
   - sum small→large if possible
   - consider compensated summation (Kahan) if accuracy matters

3) Watch out for subtracting nearly equal numbers:
   - try algebraic reformulations (stable formulas)

4) In numerical differentiation:
   - too small `h` can make results worse
   - use central differences or analytic derivatives when possible


# ✍️ Exercises

**A)** Show that `(1e16 + 1) - 1e16` is 0, but `(1e16 + 2) - 1e16` might not be. Explain why.

**B)** Modify the summation experiment:
- Replace 1e16 with 1e20 and test again
- Increase the number of 1.0 terms
Discuss what changes.

**C)** Create your own cancellation example from physics:
- e.g., compute `sqrt(r^2 + a^2) - r` for large r and small a  
Find a stable alternative.

**D)** Try central difference:
\[
f'(x)\approx \frac{f(x+h)-f(x-h)}{2h}
\]
Compare error vs `h` with the forward difference above.
